# Painting the Obama head

Paints the scan from seven angles, plus a green floral backdrop, so the paintings can be wrapped back onto the 3D head, and the cube joiner then photographs the painted head.

Charcoal drawing with bright, saturated "presidential" color, after the spirit of Kehinde Wiley's official portrait (vivid greens, royal blue, gold, rich skin tones). The style is described in the prompt, not by naming the artist.

1. **Runtime → Change runtime type → GPU** (a T4 works; an A100 is faster).
2. Run the cells in order. The guides load from [`obama_kit/`](obama_kit) in the repo, so there's nothing to upload.
3. The front view is painted first. Every other view uses it as a style reference, so the views agree.
4. Then the floral backdrop is painted, to sit behind the whole model.
5. Download `paintings.zip` at the end; the joiner wraps the paintings back onto the head.

The head is "BARACK OBAMA" by louis (https://sketchfab.com/louis), from https://sketchfab.com/3d-models/barack-obama-3010d6b0fdd843b397fa9e98437bc22a, licensed CC BY 4.0 (https://creativecommons.org/licenses/by/4.0/). We cleaned and cropped it and rendered the guides from it.

In [ ]:
!pip -q install diffusers==0.31.0 transformers==4.46.3 accelerate==1.1.1

In [ ]:
# The guides (photo, depth and mask for each view, plus the cameras) come straight from the repo.
import json, os, urllib.request
BRANCH = "cubism/obama"   # "main" once merged
BASE = f"https://raw.githubusercontent.com/rath-hy/folded-illusions/{BRANCH}/cubism/obama_kit/"
os.makedirs("kit", exist_ok=True)
urllib.request.urlretrieve(BASE + "cameras.json", "kit/cameras.json")
CAMS = json.load(open("kit/cameras.json"))
VIEWS = list(CAMS["views"])
for name in VIEWS:
    for kind in ("photo", "depth", "mask"):
        urllib.request.urlretrieve(BASE + f"{name}_{kind}.png", f"kit/{name}_{kind}.png")
print(VIEWS)

In [ ]:
import os, gc
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
from diffusers import ControlNetModel, AutoencoderKL, StableDiffusionXLControlNetImg2ImgPipeline
controlnet = ControlNetModel.from_pretrained("diffusers/controlnet-depth-sdxl-1.0", torch_dtype=torch.float16)
vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
pipe = StableDiffusionXLControlNetImg2ImgPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0", controlnet=controlnet, vae=vae,
    torch_dtype=torch.float16, variant="fp16")
try:   # style reference: the front painting guides the rest (the ViT-H version: a much smaller image encoder)
    pipe.load_ip_adapter("h94/IP-Adapter", subfolder="sdxl_models", weight_name="ip-adapter_sdxl_vit-h.safetensors",
                         image_encoder_folder="models/image_encoder")
    HAS_IP = True
except Exception as e:
    print("No IP-Adapter, views will be matched by prompt and seed only:", e)
    HAS_IP = False

# A T4 (about 15 GB) can't hold everything at once: there, idle parts wait in regular memory and move to the
# GPU only while they run. Slower, but it fits. Bigger GPUs keep everything on the GPU.
OFFLOAD = torch.cuda.get_device_properties(0).total_memory < 20e9
print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB; offload:", OFFLOAD)

def ready(p):
    """Put a pipeline on the GPU (or set up offloading) before running it."""
    gc.collect(); torch.cuda.empty_cache()
    if OFFLOAD:
        p.enable_model_cpu_offload()
    else:
        p.to("cuda")
    p.enable_vae_tiling()
    return p

## Settings
- **STRENGTH**: how far the paint departs from the photo (0.5 keeps him very close; 0.8 is loose).
- **CONTROL**: how tightly the depth map holds the shape.
- **STYLE**: how strongly the front painting steers the other views (IP-Adapter).

In [ ]:
PROMPT = ("expressive charcoal drawing portrait of Barack Obama, bold charcoal strokes, smudges and hatching on paper, "
          "the planes of the face subtly faceted with a few angular, geometric strokes, still a clear and natural likeness, "
          "bright saturated presidential colors woven into the charcoal: vivid emerald green, royal blue and gold accents, "
          "rich warm brown skin tones, crisp realist contemporary American portraiture, confident, dignified, highly detailed face")
NEGATIVE = ("photo, 3d render, plastic, blurry, deformed, extra eyes, extra nose, text, signature, watermark, frame, "
            "dull, desaturated, monochrome only, background scenery, heavily abstract, shattered, fragmented, mosaic")
# The backdrop: lush green foliage and flowers (chrysanthemums for Chicago, jasmine for Hawaii, blue lilies for
# Kenya), painted loosely, not geometric. It sits behind the whole model.
BACKDROP_PROMPT = ("dense lush wall of vivid green foliage and leaves with scattered flowers, white jasmine, "
                   "purple and white chrysanthemums, blue lilies, bright saturated color, painterly with charcoal "
                   "accents, flat frontal view, edge to edge, no people, no sky")
BACKDROP_NEGATIVE = "person, face, text, signature, watermark, frame, geometric, cubist, dull, desaturated, sky, horizon"
STRENGTH, CONTROL, STYLE = 0.62, 0.6, 0.5
STEPS, GUIDANCE, SEED = 35, 6.0, 7


In [ ]:
from PIL import Image, ImageFilter
import numpy as np
os.makedirs("paintings", exist_ok=True)

def paint(name, seed=SEED, ref=None):
    photo = Image.open(f"kit/{name}_photo.png").convert("RGB")
    depth = Image.open(f"kit/{name}_depth.png").convert("RGB")
    mask = Image.open(f"kit/{name}_mask.png").convert("L").filter(ImageFilter.MaxFilter(9))
    ready(pipe)
    kw = {}
    if HAS_IP:
        pipe.set_ip_adapter_scale(STYLE if ref is not None else 0.0)
        kw["ip_adapter_image"] = ref if ref is not None else photo
    out = pipe(prompt=PROMPT, negative_prompt=NEGATIVE, image=photo, control_image=depth,
               strength=STRENGTH, controlnet_conditioning_scale=CONTROL, num_inference_steps=STEPS,
               guidance_scale=GUIDANCE, generator=torch.Generator("cuda").manual_seed(seed), **kw).images[0]
    out = Image.composite(out, Image.new("RGB", out.size, "white"), mask)   # only the head matters
    out.save(f"paintings/{name}.png")
    return out

front = paint("front")
front

If the front isn't right, change the settings or `SEED` and rerun the cell above. Everything else follows from it.

In [ ]:
results = {"front": front}
for name in VIEWS:
    if name != "front":
        results[name] = paint(name, ref=front)
grid = Image.new("RGB", (256 * len(results), 256), "white")
for i, im in enumerate(results.values()):
    grid.paste(im.resize((256, 256)), (256 * i, 0))
grid

## The floral backdrop
One painting behind the whole model. Wherever a tile shows no head, it shows this, so head-on it reads as one continuous background. The front painting steers its palette lightly.

In [ ]:
from diffusers import StableDiffusionXLPipeline
t2i = StableDiffusionXLPipeline(**{k: v for k, v in pipe.components.items() if k in
    ("vae", "text_encoder", "text_encoder_2", "tokenizer", "tokenizer_2", "unet", "scheduler",
     "image_encoder", "feature_extractor")})
ready(t2i)
kw = {}
if HAS_IP:
    t2i.set_ip_adapter_scale(0.25)
    kw["ip_adapter_image"] = front
backdrop = t2i(prompt=BACKDROP_PROMPT, negative_prompt=BACKDROP_NEGATIVE, width=1024, height=1024,
               num_inference_steps=STEPS, guidance_scale=GUIDANCE,
               generator=torch.Generator("cuda").manual_seed(SEED), **kw).images[0]
backdrop.save("paintings/backdrop.png")
backdrop

### Redo one view

In [ ]:
# results["profile"] = paint("profile", seed=11, ref=front); results["profile"]

### Optional: sharpen up
Close-up tiles enlarge a small part of a painting, so this repaints each view at 2x with a light touch. It's slower.

In [ ]:
import os, gc
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
from diffusers import ControlNetModel, AutoencoderKL, StableDiffusionXLControlNetImg2ImgPipeline
controlnet = ControlNetModel.from_pretrained("diffusers/controlnet-depth-sdxl-1.0", torch_dtype=torch.float16)
vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
pipe = StableDiffusionXLControlNetImg2ImgPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0", controlnet=controlnet, vae=vae,
    torch_dtype=torch.float16, variant="fp16")
try:   # style reference: the front painting guides the rest (the ViT-H version: a much smaller image encoder)
    pipe.load_ip_adapter("h94/IP-Adapter", subfolder="sdxl_models", weight_name="ip-adapter_sdxl_vit-h.safetensors",
                         image_encoder_folder="models/image_encoder")
    HAS_IP = True
except Exception as e:
    print("No IP-Adapter, views will be matched by prompt and seed only:", e)
    HAS_IP = False

# A T4 (about 15 GB) can't hold everything at once: there, idle parts wait in regular memory and move to the
# GPU only while they run. Slower, but it fits. Bigger GPUs keep everything on the GPU.
OFFLOAD = torch.cuda.get_device_properties(0).total_memory < 20e9
print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB; offload:", OFFLOAD)

def ready(p):
    """Put a pipeline on the GPU (or set up offloading) before running it."""
    gc.collect(); torch.cuda.empty_cache()
    if OFFLOAD:
        p.enable_model_cpu_offload()
    else:
        p.to("cuda")
    p.enable_vae_tiling()
    return p

In [ ]:
from google.colab import files
import shutil
shutil.make_archive("paintings", "zip", "paintings")
files.download("paintings.zip")

## Flowers, placed by hand
Only a band of the backdrop ever shows: a strip down each side of the face, and bits at the top and bottom. So each flower goes exactly where it will be seen, one flower per region:
- **chrysanthemums** (Chicago) down the left
- **jasmine** (Hawaii) down the right
- **African blue lilies** (Kenya) at the top and bottom

This repaints only those spots on the leaves backdrop (`obama_kit/backdrop_leaves.png`), using SDXL's inpainting model, one flower type at a time. The masks and prompts are in `obama_kit/flowers.json`.

This section runs on its own. To skip the portrait, **restart the session**, run the install cell at the top, then start here.

In [ ]:
import os, gc, json, urllib.request, torch
from PIL import Image, ImageFilter
from google.colab import files
BRANCH = "cubism/obama"   # "main" once merged
BASE = f"https://raw.githubusercontent.com/rath-hy/folded-illusions/{BRANCH}/cubism/obama_kit/"
os.makedirs("kit", exist_ok=True)
os.makedirs("paintings", exist_ok=True)
urllib.request.urlretrieve(BASE + "flowers.json", "kit/flowers.json")
FLOWERS = json.load(open("kit/flowers.json"))
for f in ["backdrop_leaves.png"] + [v["mask"] for v in FLOWERS.values()]:
    urllib.request.urlretrieve(BASE + f, "kit/" + f)
print(list(FLOWERS))

In [ ]:
# Free the portrait models if they're loaded, then load the inpainting model.
for name in ("pipe", "t2i", "up_pipe"):
    if name in globals():
        del globals()[name]
gc.collect(); torch.cuda.empty_cache()
from diffusers import AutoPipelineForInpainting
inpaint = AutoPipelineForInpainting.from_pretrained(
    "diffusers/stable-diffusion-xl-1.0-inpainting-0.1", torch_dtype=torch.float16, variant="fp16")
if torch.cuda.get_device_properties(0).total_memory < 20e9:
    inpaint.enable_model_cpu_offload()
else:
    inpaint.to("cuda")
inpaint.enable_vae_tiling()

In [ ]:
FLOWER_STYLE = ", painterly, bright saturated color, crisp detail, seen from the front"
FLOWER_NEGATIVE = "person, face, text, signature, watermark, frame, blurry, dull, desaturated, geometric, cubist"
FLOWER_SEED, FLOWER_STEPS, FLOWER_GUIDANCE = 3, 35, 7.0

back = Image.open("kit/backdrop_leaves.png").convert("RGB")
for name, f in FLOWERS.items():
    mask = Image.open("kit/" + f["mask"]).convert("L").resize(back.size)
    out = inpaint(prompt=f["prompt"] + FLOWER_STYLE, negative_prompt=FLOWER_NEGATIVE, image=back, mask_image=mask,
                  strength=0.99, num_inference_steps=FLOWER_STEPS, guidance_scale=FLOWER_GUIDANCE,
                  generator=torch.Generator("cuda").manual_seed(FLOWER_SEED)).images[0]
    back = Image.composite(out.resize(back.size), back, mask.filter(ImageFilter.GaussianBlur(12)))
    print("painted", name)
back.save("paintings/backdrop.png")
back

If one flower type isn't right, change `FLOWER_SEED` or its prompt in `FLOWERS` and rerun the cell above. Then download the new backdrop:

In [ ]:
files.download("paintings/backdrop.png")

## Pointillist oil, face by face
The last step restyles the 25 finished face textures, the panels that get printed, so the head-on view and every tilt stay consistent.

- **`obama_faces/clean/`** are the geometric faces: the paintings wrapped on the head and cut into facets, with the faceted floral backdrop.
- **`obama_faces/dots/`** are the same faces as a pointillist underpainting, made without a GPU. Dots are small on busy parts (eyes, nose, mouth, flower edges) and large on flat facets and leaves, so the varying dot sizes are set before diffusion.

Each face starts from its dots (img2img). The clean face's edges hold the facets and features (canny ControlNet), and the diffusion turns the dots into oil daubs. Face 13, in the middle, is painted first; the rest use it as a style reference.

This section runs on its own: restart the session, run the install cell at the top, then start here.

In [ ]:
import os, gc, urllib.request, torch, cv2, numpy as np
from PIL import Image
from google.colab import files
BRANCH = "cubism/obama"   # "main" once merged
BASE = f"https://raw.githubusercontent.com/rath-hy/folded-illusions/{BRANCH}/cubism/obama_faces/"
FACES = [f"face_{i:02d}.jpg" for i in range(1, 26)]
for kind in ("clean", "dots"):
    os.makedirs(f"faces/{kind}", exist_ok=True)
    for f in FACES:
        urllib.request.urlretrieve(BASE + f"{kind}/{f}", f"faces/{kind}/{f}")
os.makedirs("faces/oil", exist_ok=True)
print(len(FACES), "faces")

In [ ]:
for name in ("pipe", "t2i", "up_pipe", "inpaint"):
    if name in globals():
        del globals()[name]
gc.collect(); torch.cuda.empty_cache()
from diffusers import ControlNetModel, AutoencoderKL, StableDiffusionXLControlNetImg2ImgPipeline
canny_net = ControlNetModel.from_pretrained("diffusers/controlnet-canny-sdxl-1.0", torch_dtype=torch.float16)
vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
oil = StableDiffusionXLControlNetImg2ImgPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0", controlnet=canny_net, vae=vae,
    torch_dtype=torch.float16, variant="fp16")
try:
    oil.load_ip_adapter("h94/IP-Adapter", subfolder="sdxl_models", weight_name="ip-adapter_sdxl_vit-h.safetensors",
                        image_encoder_folder="models/image_encoder")
    OIL_IP = True
except Exception as e:
    print("No IP-Adapter; faces are matched by prompt and seed only:", e)
    OIL_IP = False
if torch.cuda.get_device_properties(0).total_memory < 20e9:
    oil.enable_model_cpu_offload()
else:
    oil.to("cuda")
oil.enable_vae_tiling()

### Settings
- **OIL_STRENGTH**: how much the diffusion repaints the dots (0.35 keeps the dots as laid; 0.6 lets it repaint freely).
- **OIL_CONTROL**: how tightly the clean face's edges hold the facets and features.
- **OIL_STYLE**: how strongly face 13 steers the others.

In [ ]:
OIL_PROMPT = ("pointillist oil painting, Neo-Impressionist divisionism in the manner of Georges Seurat and Paul Signac, "
              "the whole surface built from thick oil paint dots and dabs of varying sizes, small dots in detailed areas, "
              "large dabs in flat areas, visible impasto and canvas texture, vivid saturated color, crisp geometric planes")
OIL_NEGATIVE = ("photo, smooth, airbrushed, digital art, 3d render, vector, blurry, text, signature, watermark, frame, "
                "dull, desaturated")
OIL_STRENGTH, OIL_CONTROL, OIL_STYLE = 0.45, 0.5, 0.5
OIL_STEPS, OIL_GUIDANCE, OIL_SEED = 30, 6.0, 5

def edges(f):
    g = cv2.cvtColor(np.asarray(Image.open(f"faces/clean/{f}").convert("RGB")), cv2.COLOR_RGB2GRAY)
    e = cv2.Canny(cv2.GaussianBlur(g, (3, 3), 0), 50, 140)
    return Image.fromarray(np.stack([e] * 3, -1))

def oil_face(f, ref=None, seed=OIL_SEED):
    kw = {}
    if OIL_IP:
        oil.set_ip_adapter_scale(OIL_STYLE if ref is not None else 0.0)
        kw["ip_adapter_image"] = ref if ref is not None else Image.open(f"faces/dots/{f}").convert("RGB")
    out = oil(prompt=OIL_PROMPT, negative_prompt=OIL_NEGATIVE, image=Image.open(f"faces/dots/{f}").convert("RGB"),
              control_image=edges(f), strength=OIL_STRENGTH, controlnet_conditioning_scale=OIL_CONTROL,
              num_inference_steps=OIL_STEPS, guidance_scale=OIL_GUIDANCE,
              generator=torch.Generator("cuda").manual_seed(seed), **kw).images[0]
    out.save(f"faces/oil/{f}", quality=93)
    return out

ref = oil_face("face_13.jpg")
ref

If face 13 isn't right, change the settings and rerun the cell above. Then paint the rest:

In [ ]:
for f in FACES:
    if f != "face_13.jpg":
        oil_face(f, ref=ref)
grid = Image.new("RGB", (5 * 200, 5 * 200), "white")
for i, f in enumerate(FACES):
    grid.paste(Image.open(f"faces/oil/{f}").resize((200, 200)), (200 * (i % 5), 200 * (i // 5)))
grid

In [ ]:
# Redo one face:
# oil_face("face_07.jpg", ref=ref, seed=11)

In [ ]:
import shutil
shutil.make_archive("faces_oil", "zip", "faces/oil")
files.download("faces_oil.zip")